In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import numpy as np
import geopandas as gpd
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.style import set_plot_style, COLORS, fa
from src.utils.plot_utils import save_figure
from src.utils.statistics import (
    normalize_year,
    normalize_integer,
    missing_by_group,
    hierarchical_impute,
)

set_plot_style()

In [2]:
DATA_PATH = "../data/raw/divar.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

### Fix DataType

In [5]:
# rooms_count
df_rooms_count = df.copy()
df_rooms_count["rooms_count"] = pd.Categorical(
    df_rooms_count["rooms_count"],
    categories=["بدون اتاق", "یک", "دو", "سه", "چهار", "پنج یا بیشتر"],
    ordered=True,
)
mapping = {"بدون اتاق": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج یا بیشتر": 5}

df_rooms_count["rooms_count"] = (
    df_rooms_count["rooms_count"].map(mapping).astype("Int8")
)
df = df_rooms_count

In [6]:
# construction_year
df_construction_year = df.copy()
df_construction_year["construction_year"] = (
    df_construction_year["construction_year"].apply(normalize_year).astype("Int16")
)
df = df_construction_year

In [7]:
# total_floors_count
df_total_floors_count = df.copy()
df_total_floors_count["total_floors_count"] = (
    df_total_floors_count["total_floors_count"]
    .apply(lambda x: normalize_integer(x, {"30+": 31, "unselect": pd.NA}))
    .astype("Int8")
)
df = df_total_floors_count

In [8]:
# unit_per_floor
df_unit_per_floor = df.copy()
df_unit_per_floor["unit_per_floor"] = (
    df_unit_per_floor["unit_per_floor"]
    .apply(lambda x: normalize_integer(x, {"more_than_8": 9, "unselect": pd.NA}))
    .astype("Int8")
)
df = df_unit_per_floor

In [9]:
# extra_person_capacity
df_extra_person_capacity = df.copy()
df_extra_person_capacity["extra_person_capacity"] = (
    df_extra_person_capacity["extra_person_capacity"]
    .apply(lambda x: normalize_integer(x, {"30+": 30}))
    .astype("Int8")
)
df = df_extra_person_capacity

In [10]:
# bool_cols
df_bool_cols = df.copy()
bool_cols = [
    "rent_to_single",
    "rent_credit_transform",
    "transformable_price",
    "has_business_deed",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "is_rebuilt",
    "has_water",
    "has_electricity",
    "has_gas",
    "has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
]

df_bool_cols[bool_cols] = df_bool_cols[bool_cols].astype("boolean")
df_bool_cols["has_balcony"] = (
    df_bool_cols["has_balcony"]
    .str.strip()
    .str.lower()
    .replace(
        {
            "true": True,
            "false": False,
            "unselect": pd.NA,
        }
    )
    .astype("boolean")
)
df = df_bool_cols

In [11]:
float_cols = [
    "rent_value",
    "price_value",
    "credit_value",
    "transformable_credit",
    "transformed_credit",
    "transformable_rent",
    "transformed_rent",
    "land_size",
    "building_size",
    "regular_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
    "location_latitude",
    "location_longitude",
    "location_radius",
]
df[float_cols] = df[float_cols].astype("float32")

In [12]:
# floor
df_floor = df.copy()
df_floor["floor"] = (
    df_floor["floor"].apply(lambda x: normalize_integer(x, {"30+": 31})).astype("Int8")
)
df = df_floor

In [13]:
# created_at_month
df["created_at_month"] = pd.to_datetime(df["created_at_month"])

In [14]:
# category_cols
df_category = df.copy()
category_with_unselect = [
    "deed_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
]
for col in category_with_unselect:
    df_category[col] = df_category[col].replace("unselect", pd.NA)

category_cols = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "user_type",
    "rent_mode",
    "rent_type",
    "price_mode",
    "credit_mode",
    "deed_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
    "property_type",
]
df_category[category_cols] = df_category[category_cols].astype("category")
df = df_category

### Invalid Values

In [15]:
# location_latitude and location_longitude
iran = gpd.read_file("../data/maps/iran_provinces.geojson")
iran_polygon = iran.union_all()
location_df = df[
    df["location_latitude"].notna() & df["location_longitude"].notna()
].copy()
location_gdf = gpd.GeoDataFrame(
    location_df,
    geometry=gpd.points_from_xy(
        location_df["location_longitude"], location_df["location_latitude"]
    ),
    crs="EPSG:4326",
)
iran_gdf = gpd.GeoDataFrame(geometry=[iran_polygon], crs="EPSG:4326")
inside = gpd.sjoin(location_gdf, iran_gdf, predicate="intersects", how="inner")
inside_index = inside.index
df = pd.concat([df[df["location_latitude"].isna()], df.loc[inside_index]]).sort_index()

In [ ]:
# building_size
df.loc[df["building_size"] > 100_000, "building_size"] = pd.NA

df["building_size"] = df.groupby("cat3_slug")["building_size"].transform(
    lambda s: s.fillna(s.median())
)

In [ ]:
# land_size
df.loc[df["land_size"] > 100_000, "land_size"] = pd.NA

# df["land_size"] = df.groupby("cat3_slug")["land_size"].transform(
#     lambda s: s.fillna(s.median())
# )

In [18]:
# price_value
df.loc[df["price_value"] == 0, "price_value"] = pd.NA
df.loc[df["price_value"] >= 1e13, "price_value"] = pd.NA

In [52]:
# rent_value
df.loc[df["transformable_rent"] >= 1e14, "transformable_rent"] = pd.NA

In [53]:
# credit_value
df.loc[df["transformable_credit"] >= 1e14, "transformable_credit"] = pd.NA

### Missing Values

In [ ]:
# IMPUTE_CONFIG = {
#     "building_size": ("median", ["cat3_slug"]),
#     "construction_year": ("median", ["cat3_slug"]),
#     "rooms_count": ("mode", ["cat3_slug"]),
#     "floor": ("mode", ["cat3_slug"]),
#     "total_floors_count": ("median", ["cat3_slug"]),
#     "unit_per_floor": ("mode", ["cat3_slug"]),
# }
# for column, (strategy, groups) in IMPUTE_CONFIG.items():
#     df[column] = hierarchical_impute(
#         df,
#         column=column,
#         groups=groups,
#         strategy=strategy,
#     )

building_size: 1 -> 0 missing (1 filled)
construction_year: 183,745 -> 0 missing (183,745 filled)
rooms_count: 153,820 -> 0 missing (153,820 filled)
floor: 457,309 -> 0 missing (457,309 filled)
total_floors_count: 694,297 -> 0 missing (694,297 filled)
unit_per_floor: 696,485 -> 0 missing (696,485 filled)


### Feature Engineering

In [22]:
# amenity_cols = [
#     "has_parking",
#     "has_warehouse",
#     "has_elevator",
#     "has_balcony",
#     "has_pool",
#     "has_jacuzzi",
#     "has_sauna",
#     "has_barbecue",
# ]
# df["amenity_count"] = df[amenity_cols].eq(True).sum(axis=1)
# df.drop(columns=amenity_cols, inplace=True)

In [23]:
# df["floor_ratio"] = df["floor"] / df["total_floors_count"]

In [24]:
# df["units_in_building"] = df["unit_per_floor"] * df["total_floors_count"]

### Selected Features

In [54]:
rent_df = df[
    (df["cat2_slug"] == "residential-rent") # 276558
    | (df["cat2_slug"] == "commercial-rent") # 76567
    | (df["cat2_slug"] == "temporary-rent") # 29903
].copy() # 383028

In [43]:
# rent_df = rent_df.sample(n=100, random_state=42).reset_index(drop=True)

In [55]:
rent_df["full_deposit"] = rent_df["transformable_credit"] + rent_df["transformable_rent"] * (100_000_000 / 3_000_000)
rent_df = rent_df[rent_df["full_deposit"].notna()]

In [56]:
features = [
    "full_deposit",
    "neighborhood_slug",
    "has_balcony",
    "is_rebuilt",
    "has_elevator",
    "location_longitude",
    "location_latitude",
    "has_parking",
    "has_warehouse",
    "cat3_slug",
    "created_at_month",
    "rooms_count",
    "floor",
    "building_size",
    "cat2_slug",
    "city_slug",
    "total_floors_count",
    "construction_year",
    "unit_per_floor",
]
rent_df = rent_df[features]

In [57]:
missing_df = rent_df.isna().mean().mul(100).round(2).reset_index()
missing_df.columns = ["feature", "missing_percent"]
missing_df = missing_df.sort_values("missing_percent", ascending=False)
missing_df

,feature,missing_percent
1,neighborhood_slug,49.41
2,has_balcony,46.10
3,is_rebuilt,39.47
4,has_elevator,33.92
5,location_longitude,33.19
6,location_latitude,33.19
7,has_parking,15.58
8,has_warehouse,15.58
0,full_deposit,0.00
9,cat3_slug,0.00


In [58]:
rent_df.head()

,full_deposit,neighborhood_slug,has_balcony,is_rebuilt,has_elevator,location_longitude,location_latitude,has_parking,has_warehouse,cat3_slug,created_at_month,rooms_count,floor,building_size,cat2_slug,city_slug,total_floors_count,construction_year,unit_per_floor
2,1.616667e+09,tohid,<NA>,False,True,51.373459,35.703865,True,True,apartment-rent,2024-10-01,3,3,132.0,residential-rent,tehran,4,1401,2
3,4.116666e+09,elahiyeh,<NA>,<NA>,True,NaN,NaN,True,False,office-rent,2024-06-01,1,4,90.0,commercial-rent,tehran,4,1400,2
5,4.500000e+08,mellirah,True,False,False,NaN,NaN,True,True,apartment-rent,2024-09-01,2,3,100.0,residential-rent,ahvaz,3,1389,2
6,6.833333e+08,NaN,<NA>,<NA>,True,NaN,NaN,False,False,office-rent,2024-11-01,2,2,80.0,commercial-rent,kermanshah,4,1395,2
11,1.200000e+09,gisha,True,True,False,51.380608,35.733952,True,True,apartment-rent,2024-08-01,2,1,91.0,residential-rent,tehran,5,1385,2


In [59]:
# Remove rows with missing target categorical values
rent_df = rent_df.dropna(subset=["cat3_slug", "city_slug"])

# Fill neighborhood
rent_df["neighborhood_slug"] = rent_df["neighborhood_slug"].fillna("Unknown")

# Fill boolean features
bool_cols = [
    "has_balcony",
    "is_rebuilt",
    "has_elevator",
    "has_parking",
    "has_warehouse",
]
rent_df[bool_cols] = rent_df[bool_cols].fillna(False)

# Fill coordinates with median of each neighborhood
rent_df["location_latitude"] = rent_df.groupby("neighborhood_slug")[
    "location_latitude"
].transform(lambda x: x.fillna(x.median()))

rent_df["location_longitude"] = rent_df.groupby("neighborhood_slug")[
    "location_longitude"
].transform(lambda x: x.fillna(x.median()))

# If an entire neighborhood had missing coordinates, fill with city median
rent_df["location_latitude"] = rent_df.groupby("city_slug")[
    "location_latitude"
].transform(lambda x: x.fillna(x.median()))

rent_df["location_longitude"] = rent_df.groupby("city_slug")[
    "location_longitude"
].transform(lambda x: x.fillna(x.median()))

# Final fallback
rent_df["location_latitude"] = rent_df["location_latitude"].fillna(
    df["location_latitude"].median()
)
rent_df["location_longitude"] = rent_df["location_longitude"].fillna(
    df["location_longitude"].median()
)

In [60]:
rent_df.info()

<class 'pandas.DataFrame'>
Index: 350368 entries, 2 to 999999
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   full_deposit        350368 non-null  float32       
 1   neighborhood_slug   350368 non-null  str           
 2   has_balcony         350368 non-null  boolean       
 3   is_rebuilt          350368 non-null  boolean       
 4   has_elevator        350368 non-null  boolean       
 5   location_longitude  350368 non-null  float32       
 6   location_latitude   350368 non-null  float32       
 7   has_parking         350368 non-null  boolean       
 8   has_warehouse       350368 non-null  boolean       
 9   cat3_slug           350368 non-null  category      
 10  created_at_month    350368 non-null  datetime64[us]
 11  rooms_count         350368 non-null  Int8          
 12  floor               350368 non-null  Int8          
 13  building_size       350368 non-null  float32 

### Create Model

In [61]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

numeric_cols = [
    "location_longitude",
    "location_latitude",
    "building_size",
    "rooms_count",
    "floor",
    "total_floors_count",
    "construction_year",
    "unit_per_floor",
    "has_balcony",
    "has_parking",
    "has_warehouse",
    "has_elevator",
    "is_rebuilt",
]

X = rent_df[numeric_cols].astype(float)
y = rent_df["full_deposit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print(r2_score(y_test, pred))

-7.284477434232883e-05


In [62]:
print(rent_df[["building_size", "full_deposit"]].corr())

               building_size  full_deposit
building_size       1.000000      0.005763
full_deposit        0.005763      1.000000


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Target
y = rent_df["full_deposit"]

# Features
X = rent_df.drop(columns=["full_deposit"])

# Convert datetime to integer
X = X.copy()
X["created_at_month"] = X["created_at_month"].astype("int64")

# Categorical columns
categorical_cols = [
    "neighborhood_slug",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="passthrough",
)

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
print(f"MAE : {mean_absolute_error(y_test, y_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")
print(f"R²  : {r2_score(y_test, y_pred):.4f}")

MAE : 2,329,422,762.30
RMSE: 28,221,130,386.30
R²  : 0.0000


In [39]:
print(y_pred[:10])
print(y_test.iloc[:10].values)

[1.87495163e+09 1.70244647e+09 2.04558173e+09 1.75869815e+09
 1.64431974e+09 1.70244647e+09 1.81682489e+09 2.04558173e+09
 1.98933005e+09 1.81682489e+09]
[2.50000000e+08 3.50000000e+08 6.03333325e+09 4.83333312e+08
 6.66666640e+07 1.26666664e+08 6.50000000e+08 5.00000000e+08
 1.73333328e+08 1.30000000e+08]
